In [3]:

# @title Обновление погоды для завершённых матчей

!pip install -q requests

import sqlite3
import requests
import logging
import time
import os
import shutil

from datetime import datetime
from pathlib import Path
from google.colab import drive


# ============================================================
# 1. ПОДКЛЮЧЕНИЕ GOOGLE DRIVE
# ============================================================

drive.mount('/content/drive')

DRIVE_FOLDER = Path('/content/drive/MyDrive/Data_Bases')
LOCAL_FOLDER = Path('/content')

DATABASES = {
    "1": "rpl.db",
    "2": "FA_Premier_League.db"
}

print("\nВ какой базе данных внести данные о погоде?")
print("1 - rpl.db")
print("2 - FA_Premier_League.db")

while True:
    choice = input("\nВведите 1 или 2: ").strip()

    if choice in DATABASES:
        break

    print("Некорректный выбор. Введите 1 или 2.")


DB_NAME = DATABASES[choice]

DRIVE_DB_PATH = DRIVE_FOLDER / DB_NAME
LOCAL_DB_PATH = LOCAL_FOLDER / DB_NAME

# Проверяем наличие папки и базы данных
if not DRIVE_FOLDER.exists():
    raise FileNotFoundError(
        f"Папка не найдена на Google Диске: {DRIVE_FOLDER}"
    )

if not DRIVE_DB_PATH.exists():
    raise FileNotFoundError(
        f"База данных не найдена: {DRIVE_DB_PATH}"
    )

# Копируем базу с Google Диска в локальную папку Colab
shutil.copy2(DRIVE_DB_PATH, LOCAL_DB_PATH)

print(f"\nВыбрана база данных: {DB_NAME}")
print(f"Источник: {DRIVE_DB_PATH}")
print("Локальная копия успешно создана.")


# ============================================================
# 2. НАСТРОЙКА ЛОГИРОВАНИЯ
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)

logger = logging.getLogger(__name__)


# ============================================================
# 3. ПОЛУЧЕНИЕ СТАТИСТИКИ БАЗЫ
# ============================================================

def get_update_statistics(db_path):

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Общее количество завершённых матчей
        cursor.execute("""
            SELECT COUNT(*)
            FROM Matches
            WHERE date(match_date) <= date('now')
        """)

        total_matches = cursor.fetchone()[0]

        # Количество матчей с заполненными данными о погоде
        cursor.execute("""
            SELECT COUNT(*)
            FROM Matches
            WHERE date(match_date) <= date('now')
              AND temperature IS NOT NULL
              AND wind_speed IS NOT NULL
              AND air_humidity IS NOT NULL
        """)

        matches_with_weather = cursor.fetchone()[0]

        # Последний матч с данными о температуре
        cursor.execute("""
            SELECT MAX(match_date)
            FROM Matches
            WHERE temperature IS NOT NULL
        """)

        last_updated_match = cursor.fetchone()[0]

        conn.close()

        return {
            'total_matches': total_matches,
            'matches_with_weather': matches_with_weather,
            'last_updated_match': last_updated_match,
            'completion_rate': (
                matches_with_weather / total_matches * 100
                if total_matches > 0 else 0
            )
        }

    except Exception as e:
        logger.error(f"Ошибка получения статистики: {e}")
        return None


# ============================================================
# 4. ОБНОВЛЕНИЕ ПОГОДЫ
# ============================================================

def get_weather_for_completed_matches():

    conn = None
    updated_count = 0
    error_count = 0
    total_matches = 0

    try:

        logger.info("=" * 60)
        logger.info("НАЧАЛО ОБНОВЛЕНИЯ ПОГОДЫ")
        logger.info(f"База данных: {DB_NAME}")
        logger.info("=" * 60)

        # Подключение к локальной копии базы
        conn = sqlite3.connect(LOCAL_DB_PATH)
        cursor = conn.cursor()

        # Находим завершённые матчи без полных данных о погоде
        query = """
            SELECT
                m.match_id,
                m.match_name,
                m.match_date,
                m.match_time,
                t1.team_name AS home_team,
                t2.team_name AS away_team,
                s.stadium_name,
                c.city_name,
                c.country_name

            FROM Matches m

            JOIN Teams t1
                ON m.team1_id = t1.team_id

            JOIN Teams t2
                ON m.team2_id = t2.team_id

            JOIN Stadiums s
                ON m.stadium_id = s.stadium_id

            JOIN Cities c
                ON s.city_id = c.city_id

            WHERE (
                m.temperature IS NULL
                OR m.wind_speed IS NULL
                OR m.air_humidity IS NULL
            )

            AND (
                date(m.match_date) < date('now')
                OR (
                    date(m.match_date) = date('now')
                    AND m.match_time <= time('now')
                )
            )

            ORDER BY
                m.match_date DESC,
                m.match_time DESC
        """

        cursor.execute(query)
        matches_to_update = cursor.fetchall()

        total_matches = len(matches_to_update)

        logger.info(
            f"Найдено матчей для обновления: {total_matches}"
        )

        if total_matches == 0:
            logger.info("Нет матчей для обновления.")
            return

        # ----------------------------------------------------
        # ГРУППИРОВКА МАТЧЕЙ ПО ГОРОДУ И ДАТЕ
        # ----------------------------------------------------

        matches_by_city_date = {}

        for match in matches_to_update:

            (
                match_id,
                match_name,
                match_date,
                match_time,
                home_team,
                away_team,
                stadium_name,
                city_name,
                country_name
            ) = match

            date_only = str(match_date)[:10]

            key = (city_name, date_only)

            if key not in matches_by_city_date:
                matches_by_city_date[key] = []

            matches_by_city_date[key].append({
                'match_id': match_id,
                'match_name': match_name,
                'match_time': match_time,
                'home_team': home_team,
                'away_team': away_team,
                'stadium_name': stadium_name,
                'country_name': country_name
            })

        logger.info(
            f"Сгруппировано городов и дат: "
            f"{len(matches_by_city_date)}"
        )

        # ----------------------------------------------------
        # КЭШ КООРДИНАТ ГОРОДОВ
        # ----------------------------------------------------

        city_coordinates_cache = {}

        total_groups = len(matches_by_city_date)
        processed_groups = 0

        # ----------------------------------------------------
        # ОБРАБОТКА ГРУПП
        # ----------------------------------------------------

        for (city_name, match_date), matches_in_group in matches_by_city_date.items():

            processed_groups += 1

            logger.info("-" * 60)
            logger.info(
                f"[{processed_groups}/{total_groups}] "
                f"{city_name}, {match_date}"
            )

            logger.info(
                f"Матчей в группе: {len(matches_in_group)}"
            )

            group_updated = 0
            group_errors = 0

            try:

                # ============================================
                # 1. ГЕОКОДИНГ ГОРОДА
                # ============================================

                if city_name not in city_coordinates_cache:

                    logger.info(
                        f"Поиск координат города: {city_name}"
                    )

                    geo_url = (
                        "https://geocoding-api.open-meteo.com/v1/search"
                    )

                    geo_response = requests.get(
                        geo_url,
                        params={
                            "name": city_name,
                            "count": 1,
                            "language": "en",
                            "format": "json"
                        },
                        timeout=10
                    )

                    geo_response.raise_for_status()

                    geo_data = geo_response.json()

                    if not geo_data.get('results'):

                        logger.warning(
                            f"Город не найден: {city_name}"
                        )

                        error_count += len(matches_in_group)
                        continue

                    lat = geo_data['results'][0]['latitude']
                    lon = geo_data['results'][0]['longitude']

                    city_coordinates_cache[city_name] = (lat, lon)

                    logger.info(
                        f"Координаты: {lat}, {lon}"
                    )

                else:

                    lat, lon = city_coordinates_cache[city_name]

                    logger.info(
                        "Используются кэшированные координаты."
                    )

                # ============================================
                # 2. ПОЛУЧЕНИЕ ПОГОДЫ ЗА ДЕНЬ МАТЧА
                # ============================================

                logger.info(
                    f"Получение погоды за {match_date}"
                )

                weather_url = (
                    "https://archive-api.open-meteo.com/v1/archive"
                )

                weather_response = requests.get(
                    weather_url,
                    params={
                        "latitude": lat,
                        "longitude": lon,
                        "start_date": match_date,
                        "end_date": match_date,
                        "hourly": (
                            "temperature_2m,"
                            "relative_humidity_2m,"
                            "wind_speed_10m"
                        ),
                        "timezone": "UTC"
                    },
                    timeout=15
                )

                weather_response.raise_for_status()

                weather_data = weather_response.json()

                if (
                    'hourly' not in weather_data
                    or not weather_data['hourly'].get('time')
                ):

                    logger.warning(
                        f"Нет погодных данных: "
                        f"{city_name}, {match_date}"
                    )

                    error_count += len(matches_in_group)
                    continue

                # ============================================
                # 3. ПОДГОТОВКА ПОЧАСОВЫХ ДАННЫХ
                # ============================================

                hourly_data = weather_data['hourly']

                time_points = hourly_data['time']
                temperatures = hourly_data['temperature_2m']
                humidities = hourly_data['relative_humidity_2m']
                wind_speeds = hourly_data['wind_speed_10m']

                weather_by_hour = {}

                for i, time_str in enumerate(time_points):

                    try:

                        hour = int(
                            time_str.split('T')[1].split(':')[0]
                        )

                        weather_by_hour[hour] = {
                            'temperature': temperatures[i],
                            'humidity': humidities[i],
                            'wind_speed': wind_speeds[i]
                        }

                    except (ValueError, IndexError, TypeError):
                        continue

                # ============================================
                # 4. ОБНОВЛЕНИЕ МАТЧЕЙ
                # ============================================

                for match_info in matches_in_group:

                    match_id = match_info['match_id']
                    match_time_str = match_info['match_time']

                    try:

                        match_hour_int = int(
                            str(match_time_str).split(':')[0]
                        )

                        match_minute_int = int(
                            str(match_time_str).split(':')[1]
                        )

                    except (ValueError, IndexError, TypeError):

                        logger.warning(
                            f"Неверное время матча {match_id}: "
                            f"{match_time_str}"
                        )

                        group_errors += 1
                        continue

                    match_time_in_minutes = (
                        match_hour_int * 60 + match_minute_int
                    )

                    best_hour_data = None
                    min_time_diff = float('inf')

                    for hour, weather_info in weather_by_hour.items():

                        weather_time_in_minutes = hour * 60

                        time_diff = abs(
                            match_time_in_minutes
                            - weather_time_in_minutes
                        )

                        # Допускается отклонение не более 30 минут
                        if time_diff <= 30:

                            if time_diff < min_time_diff:

                                min_time_diff = time_diff
                                best_hour_data = weather_info

                    if best_hour_data is None:

                        logger.warning(
                            f"Нет погоды в пределах +/- 30 минут: "
                            f"матч {match_id}, время {match_time_str}"
                        )

                        group_errors += 1
                        continue

                    weather_info = best_hour_data

                    # Проверяем, что все три показателя доступны
                    if any(
                        weather_info[key] is None
                        for key in (
                            'temperature',
                            'humidity',
                            'wind_speed'
                        )
                    ):

                        logger.warning(
                            f"Неполные погодные данные для матча "
                            f"{match_id}"
                        )

                        group_errors += 1
                        continue

                    # Обновляем погодные показатели
                    update_query = """
                        UPDATE Matches

                        SET
                            temperature = ?,
                            wind_speed = ?,
                            air_humidity = ?

                        WHERE match_id = ?
                    """

                    cursor.execute(
                        update_query,
                        (
                            int(round(weather_info['temperature'])),
                            round(weather_info['wind_speed'], 1),
                            int(round(weather_info['humidity'])),
                            match_id
                        )
                    )

                    group_updated += 1

                    logger.info(
                        f"Обновлён матч {match_id}: "
                        f"{match_info['home_team']} - "
                        f"{match_info['away_team']} | "
                        f"{weather_info['temperature']:.1f}°C | "
                        f"Ветер: {weather_info['wind_speed']:.1f} км/ч | "
                        f"Влажность: {weather_info['humidity']}%"
                    )

                # ============================================
                # 5. СОХРАНЕНИЕ ИЗМЕНЕНИЙ ГРУППЫ
                # ============================================

                conn.commit()

                updated_count += group_updated
                error_count += group_errors

                logger.info(
                    f"Группа обработана: "
                    f"{group_updated}/{len(matches_in_group)}"
                )

                if processed_groups < total_groups:
                    time.sleep(1)

            except Exception as e:

                logger.error(
                    f"Ошибка обработки группы "
                    f"{city_name}/{match_date}: {e}"
                )

                conn.rollback()

                # В случае отката изменения этой группы не учитываем
                error_count += len(matches_in_group)

                continue

        # ----------------------------------------------------
        # ИТОГОВЫЙ ОТЧЁТ
        # ----------------------------------------------------

        success_rate = (
            updated_count / total_matches * 100
            if total_matches > 0 else 0
        )

        logger.info("=" * 60)
        logger.info("ОБНОВЛЕНИЕ ПОГОДЫ ЗАВЕРШЕНО")
        logger.info("=" * 60)

        logger.info(f"База данных: {DB_NAME}")
        logger.info(f"Всего матчей: {total_matches}")
        logger.info(f"Успешно обновлено: {updated_count}")
        logger.info(f"Ошибок: {error_count}")
        logger.info(f"Успешность: {success_rate:.1f}%")

    except Exception as e:

        logger.exception(
            f"Критическая ошибка обновления погоды: {e}"
        )

        if conn:
            conn.rollback()

    finally:

        if conn:
            conn.close()


# ============================================================
# 5. ЗАПУСК И ФИНАЛЬНАЯ СТАТИСТИКА
# ============================================================

if __name__ == "__main__":

    start_time = time.time()

    # Статистика до обновления
    stats_before = get_update_statistics(LOCAL_DB_PATH)

    if stats_before:

        logger.info("=" * 60)
        logger.info("СТАТИСТИКА ДО ОБНОВЛЕНИЯ")
        logger.info(
            f"Матчи с погодой: "
            f"{stats_before['matches_with_weather']}/"
            f"{stats_before['total_matches']} "
            f"({stats_before['completion_rate']:.1f}%)"
        )

    # Запускаем обновление
    get_weather_for_completed_matches()

    # Статистика после обновления
    stats_after = get_update_statistics(LOCAL_DB_PATH)

    actual_time = time.time() - start_time

    logger.info("=" * 60)
    logger.info("ИТОГОВЫЙ ОТЧЁТ")
    logger.info("=" * 60)

    logger.info(
        f"Время выполнения: {actual_time:.1f} сек."
    )

    if stats_before and stats_after:

        logger.info("\nДО ОБНОВЛЕНИЯ:")

        logger.info(
            f"Матчи с погодой: "
            f"{stats_before['matches_with_weather']}/"
            f"{stats_before['total_matches']} "
            f"({stats_before['completion_rate']:.1f}%)"
        )

        logger.info("\nПОСЛЕ ОБНОВЛЕНИЯ:")

        logger.info(
            f"Матчи с погодой: "
            f"{stats_after['matches_with_weather']}/"
            f"{stats_after['total_matches']} "
            f"({stats_after['completion_rate']:.1f}%)"
        )

        logger.info(
            f"Последний обновлённый матч: "
            f"{stats_after['last_updated_match'] or 'N/A'}"
        )

        difference = (
            stats_after['matches_with_weather']
            - stats_before['matches_with_weather']
        )

        logger.info(
            f"Дополнительно заполнено матчей: {difference}"
        )

    # ========================================================
    # 6. СОХРАНЕНИЕ ОБНОВЛЁННОЙ БАЗЫ НА GOOGLE DRIVE
    # ========================================================

    try:

        # Закрываем все соединения перед копированием
        shutil.copy2(LOCAL_DB_PATH, DRIVE_DB_PATH)

        logger.info("=" * 60)
        logger.info("БАЗА ДАННЫХ УСПЕШНО СОХРАНЕНА")
        logger.info("=" * 60)

        logger.info(
            f"Файл: {DRIVE_DB_PATH}"
        )

        logger.info(
            "Изменения записаны в исходную базу на Google Диске."
        )

    except Exception as e:

        logger.exception(
            f"ОШИБКА СОХРАНЕНИЯ БАЗЫ НА GOOGLE DRIVE: {e}"
        )
        raise

    logger.info("=" * 60)
    logger.info("РАБОТА СКРИПТА ЗАВЕРШЕНА")
    logger.info("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

В какой базе данных внести данные о погоде?
1 - rpl.db
2 - FA_Premier_League.db

Введите 1 или 2: 1

Выбрана база данных: rpl.db
Источник: /content/drive/MyDrive/Data_Bases/rpl.db
Локальная копия успешно создана.
